# 03 - Diagnostic Test

Comprehensive diagnostic test covering 6 dimensions:
1. **Speed**: s/step, tokens/sec, padding waste
2. **Loss**: train loss convergence
3. **Overfitting**: train vs eval loss gap
4. **Gradient**: grad_norm health
5. **Generation**: sample output quality
6. **Memory**: VRAM usage

Config: 10 steps, eval at step 10

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import time
import json
import sys

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

log_path = "results/diagnostic_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Diagnostic test started")

In [ ]:
import yaml

with open("configs/diagnostic.yaml") as f:
    config = yaml.safe_load(f)

log("Config loaded:")
log(f"  Model: {config['model']['name']}")
log(f"  Max seq length: {config['model']['max_seq_length']}")
log(f"  LoRA r={config['lora']['r']}, alpha={config['lora']['alpha']}")
log(f"  Batch: {config['training']['batch_size']}")
log(f"  Grad accum: {config['training']['gradient_accumulation_steps']}")
log(f"  Max steps: {config['training']['max_steps']}")

In [ ]:
log("=== Data Pipeline ===")
t0 = time.time()

if not os.path.exists("data/splits/train.json"):
    log("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

t1 = time.time()
log(f"Data pipeline: {t1-t0:.1f}s")

with open("data/splits/train.json") as f:
    train_data = json.load(f)
with open("data/splits/val.json") as f:
    val_data = json.load(f)

log(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
import torch

log("=== 6. Memory: Model Loading ===")
log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

t0 = time.time()

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["model"]["name"],
    max_seq_length=config["model"]["max_seq_length"],
    load_in_4bit=config["model"]["load_in_4bit"],
)

t1 = time.time()
log(f"Model loading: {t1-t0:.1f}s")
log(f"VRAM after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
log("=== LoRA Setup ===")
t0 = time.time()

model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora"]["r"],
    target_modules=config["lora"]["target_modules"],
    lora_alpha=config["lora"]["alpha"],
    lora_dropout=config["lora"]["dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

t1 = time.time()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log(f"LoRA setup: {t1-t0:.1f}s")
log(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
log(f"VRAM after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
log("=== Dataset Preparation ===")
t0 = time.time()

from datasets import Dataset

def load_split(path):
    with open(path) as f:
        data = json.load(f)
    texts = []
    for ex in data:
        messages = []
        for turn in ex["conversations"]:
            role = "assistant" if turn["from"] == "gpt" else turn["from"]
            messages.append({"role": role, "content": turn["value"]})
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append({"text": text})
    return Dataset.from_list(texts)

train_dataset = load_split("data/splits/train.json")
val_dataset = load_split("data/splits/val.json")

t1 = time.time()
log(f"Dataset prep: {t1-t0:.1f}s")
log(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# 1. Speed: Token length analysis
lengths = [tokenizer(train_dataset[i]["text"], return_tensors="pt")["input_ids"].shape[1] for i in range(min(100, len(train_dataset)))]
avg_length = sum(lengths) // len(lengths)
max_seq = config["model"]["max_seq_length"]
padding_ratio = avg_length / max_seq

log(f"\n1. Speed: Token Length Analysis:")
log(f"  Token lengths: min={min(lengths)}, max={max(lengths)}, avg={avg_length}")
log(f"  Max seq length: {max_seq}")
log(f"  Padding ratio: {padding_ratio:.1%} (avg_length / max_seq_length)")
if padding_ratio < 0.3:
    log(f"  ⚠️  Heavy padding waste ({1-padding_ratio:.1%} wasted)")
    log(f"  💡 Consider reducing max_seq_length to {avg_length * 2}")
else:
    log(f"  ✅ Padding ratio is acceptable")

In [ ]:
log("=== 1. Speed & 2. Loss & 3. Overfitting & 4. Gradient: Training Test ===")

from trl import SFTTrainer, SFTConfig

# Use eval at step 10 for overfitting check
training_args = SFTConfig(
    output_dir="outputs/diagnostic",
    per_device_train_batch_size=config["training"]["batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    max_steps=config["training"]["max_steps"],
    learning_rate=config["training"]["learning_rate"],
    lr_scheduler_type=config["training"]["scheduler"],
    optim=config["training"]["optim"],
    max_seq_length=config["model"]["max_seq_length"],
    fp16=config["training"]["fp16"],
    logging_steps=1,
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=10,  # Eval at step 10 for overfitting check
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

log("Starting 10-step training (with eval at step 10)...")
t0 = time.time()
trainer.train()
t1 = time.time()

total_time = t1 - t0
time_per_step = total_time / config["training"]["max_steps"]

# 1. Speed: Calculate throughput
batch_size = config["training"]["batch_size"]
tokens_per_step = batch_size * max_seq
throughput = tokens_per_step / time_per_step

log(f"\n1. Speed Analysis:")
log(f"  Time per step: {time_per_step:.2f}s")
log(f"  Tokens per step: {tokens_per_step}")
log(f"  Throughput: {throughput:.0f} tokens/sec")
log(f"  Effective throughput (no padding): {batch_size * avg_length / time_per_step:.0f} tokens/sec")

# 2. Loss analysis
log_history = trainer.state.log_history
train_losses = [l["loss"] for l in log_history if "loss" in l]
eval_losses = [l["eval_loss"] for l in log_history if "eval_loss" in l]
grad_norms = [l["grad_norm"] for l in log_history if "grad_norm" in l]

if train_losses:
    log(f"\n2. Loss Analysis:")
    log(f"  Initial train loss: {train_losses[0]:.4f}")
    log(f"  Final train loss: {train_losses[-1]:.4f}")
    log(f"  Loss change: {train_losses[0] - train_losses[-1]:.4f}")
    if train_losses[-1] < train_losses[0]:
        log("  ✅ Loss is decreasing")
    else:
        log("  ❌ Loss is NOT decreasing")

# 3. Overfitting check
if eval_losses and train_losses:
    gap = eval_losses[-1] - train_losses[-1]
    log(f"\n3. Overfitting Analysis:")
    log(f"  Final train loss: {train_losses[-1]:.4f}")
    log(f"  Final eval loss: {eval_losses[-1]:.4f}")
    log(f"  Gap (eval - train): {gap:.4f}")
    if gap > 0.5:
        log("  ❌ Large gap - potential overfitting")
    elif gap > 0.2:
        log("  ⚠️  Moderate gap - monitor during full training")
    else:
        log("  ✅ Gap is small - no overfitting detected")
else:
    log("\n3. Overfitting Analysis: No eval results (check eval config)")

# 4. Gradient analysis
if grad_norms:
    log(f"\n4. Gradient Analysis:")
    log(f"  Avg grad_norm: {sum(grad_norms)/len(grad_norms):.4f}")
    log(f"  Max grad_norm: {max(grad_norms):.4f}")
    log(f"  Min grad_norm: {min(grad_norms):.4f}")
    if max(grad_norms) > 10:
        log("  ❌ Gradient explosion detected")
    elif min(grad_norms) < 0.001:
        log("  ❌ Gradient vanishing detected")
    else:
        log("  ✅ Gradients are healthy")

In [ ]:
log("\n=== 5. Generation: Sample Output ===")

FastLanguageModel.for_inference(model)

test_prompts = [
    "Analyze the sentiment: Tesla shares surged 12% after record Q3 deliveries",
    "What are the key risks of investing in emerging market bonds?",
]

generation_ok = True
for i, prompt in enumerate(test_prompts):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    log(f"\nPrompt {i+1}: {prompt[:60]}...")
    log(f"Response: {response[:150]}...")
    # Basic quality check
    if len(response) < 10:
        generation_ok = False
        log("  ⚠️  Response too short")

if generation_ok:
    log("\n✅ Generation test passed")
else:
    log("\n❌ Generation test failed")

In [ ]:
log("\n=== 6. Memory: Summary ===")
peak_vram = torch.cuda.max_memory_allocated() / 1e9
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_usage = peak_vram / total_vram

log(f"Peak VRAM: {peak_vram:.2f} GB")
log(f"VRAM limit: {total_vram:.1f} GB")
log(f"VRAM usage: {vram_usage:.1%}")

if vram_usage > 0.9:
    log("❌ VRAM usage too high (>90%)")
elif vram_usage > 0.8:
    log("⚠️  VRAM usage high (>80%)")
else:
    log("✅ VRAM usage is safe")

In [ ]:
log("\n" + "=" * 60)
log("DIAGNOSTIC SUMMARY (6 Dimensions)")
log("=" * 60)

# Build results for all 6 dimensions
results = {
    "1_speed": {
        "time_per_step": time_per_step,
        "throughput_tokens_sec": throughput,
        "padding_ratio": padding_ratio,
        "pass": time_per_step < 15,
    },
    "2_loss": {
        "initial": train_losses[0] if train_losses else None,
        "final": train_losses[-1] if train_losses else None,
        "pass": train_losses[-1] < train_losses[0] if train_losses else False,
    },
    "3_overfitting": {
        "train_loss": train_losses[-1] if train_losses else None,
        "eval_loss": eval_losses[-1] if eval_losses else None,
        "gap": eval_losses[-1] - train_losses[-1] if (eval_losses and train_losses) else None,
        "pass": (eval_losses[-1] - train_losses[-1] < 0.5) if (eval_losses and train_losses) else False,
    },
    "4_gradient": {
        "avg": sum(grad_norms)/len(grad_norms) if grad_norms else None,
        "max": max(grad_norms) if grad_norms else None,
        "pass": all(0.001 < g < 10 for g in grad_norms) if grad_norms else False,
    },
    "5_generation": {
        "pass": generation_ok,
    },
    "6_memory": {
        "peak_gb": peak_vram,
        "usage_pct": vram_usage,
        "pass": vram_usage < 0.9,
    },
}

# Print summary
for dim, r in results.items():
    status = "✅" if r.get("pass") else "❌"
    log(f"{status} {dim}: {r}")

# Overall verdict
all_pass = all(r.get("pass", False) for r in results.values())
log(f"\nOverall: {"✅ ALL PASSED" if all_pass else "❌ SOME FAILED"}")

if all_pass:
    log("Ready for full training!")
else:
    log("Fix issues before full training.")

# Save results
results_path = "results/diagnostic_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
log(f"\nResults saved to: {results_path}")
log(f"Log saved to: {log_path}")